In [11]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/rechecked-finaldatasetforcausal/finaldatasetwithcourseinformationforcausalanalysis.csv


In [21]:
pip install econml

Note: you may need to restart the kernel to use updated packages.


In [22]:
from sklearn.model_selection import KFold
import pandas as pd
import numpy as np
from scipy import stats
from tqdm import tqdm
import re
from econml.dml import CausalForestDML
from sklearn.preprocessing import StandardScaler
print("Loading data...")
df = pd.read_csv('/kaggle/input/rechecked-finaldatasetforcausal/finaldatasetwithcourseinformationforcausalanalysis.csv')

def clean_name(name):
    return re.sub(r'[^a-zA-Z0-9]', '_', name)


df_grouped = df.groupby('id_x').agg({
    'rate': 'mean',
    'group_category': 'first',
    'predicted_aspect': lambda x: list(set(x)),
    'predicted_sentiment_ensemble': lambda x: list(x),
    'price': 'first',
    'is_paid': 'first',
    'num_lectures': 'first'
}).reset_index()

Loading data...


In [23]:
aspects = df['predicted_aspect'].unique()
for aspect in tqdm(aspects, desc="Creating sentiment columns"):
    clean_aspect = clean_name(aspect)
    sentiment_col = f'sentiment_{clean_aspect}'
    indicator_col = f'has_{clean_aspect}'
    
    df_grouped[sentiment_col] = df_grouped.apply(
        lambda row: np.mean([2 if s == 'Positive' else 1 if s == 'Neutral' else 0 
                             for a, s in zip(row['predicted_aspect'], row['predicted_sentiment_ensemble']) if a == aspect]) 
        if aspect in row['predicted_aspect'] else np.nan, 
        axis=1
    )
    
    df_grouped[indicator_col] = df_grouped[sentiment_col].notna().astype(int)
    df_grouped[sentiment_col] = df_grouped[sentiment_col].fillna(-1)

Creating sentiment columns: 100%|██████████| 6/6 [01:00<00:00, 10.06s/it]


In [24]:
from sklearn.model_selection import KFold
import pandas as pd
import numpy as np
from scipy import stats
from tqdm import tqdm
import re
from econml.dml import CausalForestDML
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
import gc

def causal_forest_analysis(df, target_aspect, aspects, category=None, n_folds=5):
    clean_target = clean_name(target_aspect)
    
    if category:
        df = df[df['group_category'] == category]
    
    X_columns = []
    for asp in aspects:
        if asp != target_aspect:
            clean_asp = clean_name(asp)
            X_columns.extend([f'sentiment_{clean_asp}', f'has_{clean_asp}'])
    X_columns += ['price', 'num_lectures', 'is_paid']
    
    X = df[X_columns]
    y = df['rate']
    T = df[f'sentiment_{clean_target}']
    
    mask = df[f'has_{clean_target}'] == 1
    X = X[mask]
    y = y[mask]
    T = T[mask]
    
    scale_columns = ['price', 'num_lectures'] + [col for col in X.columns if col.startswith('sentiment_')]
    
    scaler = StandardScaler()
    X_scaled = X.copy()
    X_scaled[scale_columns] = scaler.fit_transform(X[scale_columns])
    model_y = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
    model_t = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
    
    cf_model = CausalForestDML(
        model_y = model_y,
        model_t = model_t,
        n_estimators=100,
        min_samples_leaf=5,
        max_depth=4,
        cv=n_folds,
        verbose=0,
        random_state=42
    )
    
    cf_model.fit(y, T, X=X_scaled)
    
    te_pred = cf_model.effect(X_scaled)
    
    ate = np.mean(te_pred)
    ate_std = np.std(te_pred)
    effect_range = (np.min(te_pred), np.max(te_pred))
    feature_importance = dict(zip(X_columns, cf_model.feature_importances_))
    
    return {
        'aspect': target_aspect,
        'category': category,
        'ate': ate,
        'ate_std': ate_std,
        'effect_range': effect_range,
        'feature_importance': feature_importance,
        'sample_size': len(X)
    }

In [25]:
categories = df_grouped['group_category'].unique()


In [26]:
for category in categories:
    print(f"\n\nAnalyzing category: {category}")
    category_results = {}
    
    for aspect in aspects:
        result = causal_forest_analysis(df_grouped, aspect, aspects, category, n_folds=5)
        if result:
            category_results[aspect] = result
    
    # แสดงผลลัพธ์สำหรับ category นี้ทันที
    print(f"\nResults for Category: {category}")
    for aspect, result in category_results.items():
        print(f"\nAspect: {aspect}")
        print(f"ATE: {result['ate']:.4f} ± {1.96 * result['ate_std']:.4f}")
        print(f"Effect range: {result['effect_range']}")
        print(f"Sample size: {result['sample_size']}")
        print("Top 5 Feature Importance:")
        for feature, importance in sorted(result['feature_importance'].items(), key=lambda x: x[1], reverse=True)[:5]:
            print(f"{feature}: {importance:.4f}")
            
    del category_results
    gc.collect()



Analyzing category: Technology and IT

Results for Category: Technology and IT

Aspect: Instructor
ATE: 1.2318 ± 0.5766
Effect range: (0.3338240530648599, 1.471955295979111)
Sample size: 133451
Top 5 Feature Importance:
sentiment_Assignment___Tutorial: 0.3875
has_Assignment___Tutorial: 0.3646
sentiment_Design___Material: 0.1466
has_Platform___Technical: 0.0483
sentiment_Platform___Technical: 0.0295

Aspect: Assignment & Tutorial
ATE: 1.0532 ± 0.6039
Effect range: (0.38967396632314577, 1.3492748985867904)
Sample size: 56096
Top 5 Feature Importance:
sentiment_Instructor: 0.5534
has_Instructor: 0.2999
sentiment_Design___Material: 0.1002
sentiment_Platform___Technical: 0.0303
sentiment_Skill___Career: 0.0068

Aspect: Design & Material
ATE: 1.0689 ± 0.6176
Effect range: (0.36581109500299813, 1.3332622093526514)
Sample size: 67041
Top 5 Feature Importance:
sentiment_Instructor: 0.4795
has_Instructor: 0.3874
sentiment_Platform___Technical: 0.0471
has_Platform___Technical: 0.0336
sentiment_

In [18]:
# from sklearn.model_selection import KFold
# import pandas as pd
# import numpy as np
# from scipy import stats
# from tqdm import tqdm
# import re
# from econml.dml import CausalForestDML
# from sklearn.preprocessing import StandardScaler
# from sklearn.ensemble import RandomForestRegressor
# import gc

# def causal_forest_analysis(df, target_aspect, aspects, category=None, n_folds=5):
#     clean_target = clean_name(target_aspect)
    
#     if category:
#         df = df[df['group_category'] == category]
    
#     X_columns = []
#     for asp in aspects:
#         if asp != target_aspect:
#             clean_asp = clean_name(asp)
#             X_columns.extend([f'sentiment_{clean_asp}', f'has_{clean_asp}'])
#     X_columns += ['price', 'num_lectures', 'is_paid']
    
#     X = df[X_columns]
#     y = df['rate']
#     T = df[f'sentiment_{clean_target}']
    
#     mask = df[f'has_{clean_target}'] == 1
#     X = X[mask]
#     y = y[mask]
#     T = T[mask]
    
#     scale_columns = ['price', 'num_lectures'] + [col for col in X.columns if col.startswith('sentiment_')]
#     no_scale_columns = [col for col in X.columns if col not in scale_columns]
    
#     kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    
#     ate_results = []
#     ate_std_results = []
#     effect_range_results = []
#     feature_importance_results = []
    
#     for train_index, test_index in kf.split(X):
#         X_train, X_test = X.iloc[train_index], X.iloc[test_index]
#         y_train, y_test = y.iloc[train_index], y.iloc[test_index]
#         T_train, T_test = T.iloc[train_index], T.iloc[test_index]
        
#         scaler = StandardScaler()
#         X_train_scaled = X_train.copy()
#         X_test_scaled = X_test.copy()
#         X_train_scaled[scale_columns] = scaler.fit_transform(X_train[scale_columns])
#         X_test_scaled[scale_columns] = scaler.transform(X_test[scale_columns])
        
#         # กำหนดโมเดลสำหรับ Y และ T อย่างชัดเจน
#         model_y = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)
#         model_t = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)
        
#         cf_model = CausalForestDML(
#             model_y=model_y,
#             model_t=model_t,
#             n_estimators=200,
#             min_samples_leaf=5,
#             max_depth=10,
#             verbose=0,
#             random_state=42
#         )
#         cf_model.fit(y_train, T_train, X=X_train_scaled)
        
#         te_pred = cf_model.effect(X_test_scaled)
        
#         ate_results.append(np.mean(te_pred))
#         ate_std_results.append(np.std(te_pred))
#         effect_range_results.append((np.min(te_pred), np.max(te_pred)))
#         feature_importance_results.append(cf_model.feature_importances_)
    
#     ate = np.mean(ate_results)
#     ate_std = np.mean(ate_std_results)
#     effect_range = (
#         np.mean([er[0] for er in effect_range_results]),
#         np.mean([er[1] for er in effect_range_results])
#     )
#     feature_importance = dict(zip(X_columns, np.mean(feature_importance_results, axis=0)))
    
#     return {
#         'aspect': target_aspect,
#         'category': category,
#         'ate': ate,
#         'ate_std': ate_std,
#         'effect_range': effect_range,
#         'feature_importance': feature_importance,
#         'sample_size': len(X)
#     }

Sensitivity Analysis

In [29]:
def causal_forest_analysis(df, target_aspect, aspects, category=None, cf_params=None, rf_params=None):
    clean_target = clean_name(target_aspect)

    if category:
        df = df[df['group_category'] == category]

    X_columns = []
    for asp in aspects:
        if asp != target_aspect:
            clean_asp = clean_name(asp)
            X_columns.extend([f'sentiment_{clean_asp}', f'has_{clean_asp}'])
    X_columns += ['price', 'num_lectures', 'is_paid']

    X = df[X_columns]
    y = df['rate']
    T = df[f'sentiment_{clean_target}']

    mask = df[f'has_{clean_target}'] == 1
    X = X[mask]
    y = y[mask]
    T = T[mask]

    scale_columns = ['price', 'num_lectures'] + [col for col in X.columns if col.startswith('sentiment_')]

    scaler = StandardScaler()
    X_scaled = X.copy()
    X_scaled[scale_columns] = scaler.fit_transform(X[scale_columns])

    model_y = RandomForestRegressor(**rf_params)
    model_t = RandomForestRegressor(**rf_params)

    cf_model = CausalForestDML(
        model_y=model_y,
        model_t=model_t,
        **cf_params
    )
    cf_model.fit(y, T, X=X_scaled)

    te_pred = cf_model.effect(X_scaled)

    ate = np.mean(te_pred)
    ate_std = np.std(te_pred)
    effect_range = (np.min(te_pred), np.max(te_pred))
    feature_importance = dict(zip(X_columns, cf_model.feature_importances_))

    return {
        'aspect': target_aspect,
        'category': category,
        'ate': ate,
        'ate_std': ate_std,
        'effect_range': effect_range,
        'feature_importance': feature_importance,
        'sample_size': len(X)
    }

def sensitivity_analysis(df_grouped, aspects, categories):
    parameter_sets = [
        {
            'name': 'Initial',
            'cf_params': {
                'n_estimators': 100,
                'min_samples_leaf': 5,
                'max_depth': 4,
                'cv': 5,
                'verbose': 0,
                'random_state': 42
            },
            'rf_params': {
                'n_estimators': 100,
                'max_depth': 5,
                'random_state': 42
            }
        },
        {
            'name': 'More Complex',
            'cf_params': {
                'n_estimators': 1000,
                'min_samples_leaf': 10,
                'max_depth': 10,
                'cv': 5,
                'verbose': 0,
                'random_state': 42
            },
            'rf_params': {
                'n_estimators': 100,
                'max_depth': 5,
                'random_state': 42
            }
        },
        {
            'name': 'More Simple',
            'cf_params': {
                'n_estimators': 52,
                'min_samples_leaf': 5,
                'max_depth': 4,
                'cv': 5,
                'verbose': 0,
                'random_state': 42
            },
            'rf_params': {
                'n_estimators': 100,
                'max_depth': 5,
                'random_state': 42
            }
        }
    ]

    results = {}

    for params in parameter_sets:
        print(f"\n\nRunning analysis with {params['name']} parameters")
        params_results = {}

        for category in categories:
            print(f"\nAnalyzing category: {category}")
            category_results = {}

            for aspect in aspects:
                result = causal_forest_analysis(df_grouped, aspect, aspects, category, 
                                                cf_params=params['cf_params'], 
                                                rf_params=params['rf_params'])
                if result:
                    category_results[aspect] = result

            params_results[category] = category_results

        results[params['name']] = params_results

    return results



In [30]:
categories = df_grouped['group_category'].unique()
sensitivity_results = sensitivity_analysis(df_grouped, aspects, categories)

for params_name, params_results in sensitivity_results.items():
    print(f"\n\nResults for {params_name} parameters:")
    for category, category_results in params_results.items():
        print(f"\nCategory: {category}")
        for aspect, result in category_results.items():
            print(f"\nAspect: {aspect}")
            print(f"ATE: {result['ate']:.4f} ± {1.96 * result['ate_std']:.4f}")
            print(f"Effect range: {result['effect_range']}")
            print(f"Sample size: {result['sample_size']}")
            print("Top 5 Feature Importance:")
            for feature, importance in sorted(result['feature_importance'].items(), key=lambda x: x[1], reverse=True)[:5]:
                print(f"{feature}: {importance:.4f}")
    gc.collect()



Running analysis with Initial parameters

Analyzing category: Technology and IT

Analyzing category: Business and Finance

Analyzing category: Creative and Design

Analyzing category: Education and Teaching

Analyzing category: Personal Development and Lifestyle

Analyzing category: Marketing and Productivity

Analyzing category: Health and Fitness


Running analysis with More Complex parameters

Analyzing category: Technology and IT

Analyzing category: Business and Finance

Analyzing category: Creative and Design

Analyzing category: Education and Teaching

Analyzing category: Personal Development and Lifestyle

Analyzing category: Marketing and Productivity

Analyzing category: Health and Fitness


Running analysis with More Simple parameters

Analyzing category: Technology and IT

Analyzing category: Business and Finance

Analyzing category: Creative and Design

Analyzing category: Education and Teaching

Analyzing category: Personal Development and Lifestyle

Analyzing category: